# Pandas Analyzing Data

> 📘 **Python Mastery** · Module 11 — Pandas · Lesson 4/7

Before any fancy modeling, analysts answer simple questions: What does the data look like? Which values repeat? What is typical? What moves together? This lesson is the first-look toolkit — the commands you will type within minutes of loading any new table.

## 🎯 Learning Objectives

- **Profile** a fresh dataset with `head`, `info()` and `describe()` — including non-numeric columns.
- **Summarize** categorical columns with `value_counts()`, `unique()` and `nunique()`.
- **Compute** column statistics (`mean`, `median`, `std`, `sum`, ...) and know when `numeric_only=True` is required.
- **Distinguish** `count()` from `size`, and explain why they disagree on messy data.
- **Rank and sort** with multi-key `sort_values()` plus `nlargest` / `nsmallest`.
- **Measure relationships** with `df.corr(numeric_only=True)` and filter rows declaratively with `.query()`.

## 1. Meet the Dataset

Seven students from around Bangladesh: their age, home city, weekly study hours, and final exam score. Each code cell below rebuilds this same table so the notebook runs cleanly top-to-bottom — in your own notebooks you would load once and reuse.

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":        ["Sarah", "Rafi", "Nabila", "Karim", "Mim", "Tanvir", "Jony"],
    "age":         [22, 25, 21, 23, 20, 24, 22],
    "city":        ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna", "Dhaka", "Sylhet"],
    "study_hours": [2, 4, 1, 5, 3, 4, 2],
    "score":       [88, 92, 79, 95, 84, 90, 74],
})
print(students)

## 2. The First-Look Toolkit: `head`, `info`, `describe`

Three calls, three altitudes: `head()` shows *specimens* of rows, `info()` shows *structure* (types, missing counts), and `describe()` shows *statistics*.

By default `describe()` covers numeric columns only. Pass `include="all"` to also profile text columns — those report `unique` (how many distinct values), `top` (the most frequent one) and `freq` (its count).

> 🔍 **Under the Hood:** `describe()` is not doing anything magical — it calls NumPy's percentile machinery on each numeric block (`25%`, `50%`, `75%`) and a hash-table count for `freq`. It just assembles the answers into one tidy table, which is why adding `include="all"` mixes numbers with strings in the same frame and produces those `NaN` gaps where a statistic does not apply.

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":        ["Sarah", "Rafi", "Nabila", "Karim", "Mim", "Tanvir", "Jony"],
    "age":         [22, 25, 21, 23, 20, 24, 22],
    "city":        ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna", "Dhaka", "Sylhet"],
    "study_hours": [2, 4, 1, 5, 3, 4, 2],
    "score":       [88, 92, 79, 95, 84, 90, 74],
})

print(students.head(3))          # first 3 rows
students.info()                  # structure: dtypes + non-null counts
print()
print(students.describe().round(2))            # numeric profile
print()
print(students.describe(include="all").round(2))   # text columns included

## 3. Frequencies and Distinct Values: `value_counts`, `unique`, `nunique`

Categorical columns need different questions than numeric ones: *which values exist?* (`unique`) — *how many are there?* (`nunique`) — *how often does each appear?* (`value_counts`). Add `normalize=True` to turn raw counts into proportions (they sum to 1).

**Syntax:**

```python
s.value_counts(normalize=False, ascending=False)   # frequency table (sorted, biggest first)
s.unique()        # array of distinct values, order of appearance
s.nunique()       # NUMBER of distinct values
df.nunique()      # per-column version works on whole DataFrames too
```

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":        ["Sarah", "Rafi", "Nabila", "Karim", "Mim", "Tanvir", "Jony"],
    "age":         [22, 25, 21, 23, 20, 24, 22],
    "city":        ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna", "Dhaka", "Sylhet"],
    "study_hours": [2, 4, 1, 5, 3, 4, 2],
    "score":       [88, 92, 79, 95, 84, 90, 74],
})

print(students["city"].value_counts())          # Dhaka wins with 3 students
print()
print(students["city"].value_counts(normalize=True).round(2))   # as fractions of 7
print()
print(students["city"].unique(), "| distinct cities:", students["city"].nunique())
print(students[["age", "city"]].nunique())      # per-column distinct counts

## 4. Column Statistics — and the `numeric_only=` Rule

One call computes a statistic down every column: `mean()`, `median()`, `min()`, `max()`, `std()`, `sum()`.

⚠️ In pandas 3, calling these on a table with **any text column raises** `TypeError: Cannot perform reduction 'mean' with string dtype`. Pass **`numeric_only=True`** and pandas quietly skips non-numeric columns. On a single Series no flag is needed — a Series has one dtype by definition.

**Syntax:**

```python
df.mean(numeric_only=True)     # one mean per numeric column -> Series
s.mean()                       # single column needs no flag
s.quantile(0.75)               # any percentile you like
```

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":        ["Sarah", "Rafi", "Nabila", "Karim", "Mim", "Tanvir", "Jony"],
    "age":         [22, 25, 21, 23, 20, 24, 22],
    "city":        ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna", "Dhaka", "Sylhet"],
    "study_hours": [2, 4, 1, 5, 3, 4, 2],
    "score":       [88, 92, 79, 95, 84, 90, 74],
})

print(students[["age", "study_hours", "score"]].mean())        # chosen columns only
print()
print(students.mean(numeric_only=True).round(1))               # whole frame, numbers kept
print()
print("median score:", students["score"].median(),
      "| std:", round(students["score"].std(), 2),
      "| total study hours:", students["study_hours"].sum())
print("best score:", students["score"].max(), "| 75th percentile:", students["score"].quantile(0.75))

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name": ["Sarah", "Rafi", "Nabila"],
    "age":  [22, 25, 21],
    "score": [88, 92, 79],
})

# pandas 3 refuses to average across text columns -- explicit beats silent:
try:
    students.mean()
except TypeError as e:
    print("TypeError:", e)
print()
print(students.mean(numeric_only=True))   # <- the fix

## 5. `count()` vs `size` — A Subtle but Important Pair

- **`df.count()`** — a *method*: per column, how many **non-missing** values.
- **`df.size`** — an *attribute* (no parentheses): total cells in the table, rows × columns, missing or not.

On clean data every column counts equally and `sum(df.count()) == df.size`. The moment data has holes, they diverge — which makes their difference a quick missing-data detector.

**Example:**

In [ ]:
import pandas as pd

messy = pd.DataFrame({
    "name":  ["Sarah", "Rafi", None,     "Karim"],
    "score": [88,      None,  79,        95],
})

print(messy)                       # two holes in a 4x2 table (8 cells)
print()
print("count per column:", messy.count().to_dict())   # only present values
print("size:", messy.size, "| non-null total:", int(messy.count().sum()))
print("missing cells:", messy.size - int(messy.count().sum()))

## 6. Sorting: `sort_values` with Multiple Keys

Sorting answers "who is first?". Pass a **list** of columns to break ties hierarchically, and a matching `ascending=` list to control each key's direction. Like most pandas methods it returns a sorted *copy*.

**Syntax:**

```python
df.sort_values("col")                                   # ascending by one column
df.sort_values(by=["city", "score"], ascending=[True, False])   # per-key directions
```

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":        ["Sarah", "Rafi", "Nabila", "Karim", "Mim", "Tanvir", "Jony"],
    "age":         [22, 25, 21, 23, 20, 24, 22],
    "city":        ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna", "Dhaka", "Sylhet"],
    "study_hours": [2, 4, 1, 5, 3, 4, 2],
    "score":       [88, 92, 79, 95, 84, 90, 74],
})

by_city = students.sort_values(by=["city", "score"], ascending=[True, False])
print(by_city[["city", "name", "score"]])   # cities A->Z; within a city, best score first

## 7. Top-N Without Sorting Everything: `nlargest` / `nsmallest`

When you only want the extremes, `nlargest(n, col)` / `nsmallest(n, col)` beat `sort_values().head(n)` — they are clearer *and* faster, because pandas keeps just n candidates instead of fully ordering the column.

**Syntax:**

```python
df.nlargest(3, "score")          # top 3 rows by score
df.nsmallest(2, ["age", "score"])   # ties broken by the second column
```

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":        ["Sarah", "Rafi", "Nabila", "Karim", "Mim", "Tanvir", "Jony"],
    "age":         [22, 25, 21, 23, 20, 24, 22],
    "city":        ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna", "Dhaka", "Sylhet"],
    "study_hours": [2, 4, 1, 5, 3, 4, 2],
    "score":       [88, 92, 79, 95, 84, 90, 74],
})

print(students.nlargest(3, "score")[["name", "score"]])     # the podium
print()
print(students.nsmallest(2, "score")[["name", "score"]])    # who needs tutoring

## 8. Do Columns Move Together? `df.corr()`

The **correlation coefficient** measures how strongly two numeric columns move together, on a scale from -1 to +1: near +1 means "rise together", near -1 means "opposite directions", near 0 means "no linear relation".

`df.corr()` returns every numeric column paired with every other. In pandas 3 you must pass `numeric_only=True` whenever text columns are present.

> 🔍 **Under the Hood:** Pearson correlation is covariance divided by the product of standard deviations — a pure NumPy matrix operation over the numeric blocks, so it costs almost nothing even on millions of rows. It only captures *linear* relationships: two columns can be tightly related in a curved way and still score near 0.

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":        ["Sarah", "Rafi", "Nabila", "Karim", "Mim", "Tanvir", "Jony"],
    "age":         [22, 25, 21, 23, 20, 24, 22],
    "city":        ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna", "Dhaka", "Sylhet"],
    "study_hours": [2, 4, 1, 5, 3, 4, 2],
    "score":       [88, 92, 79, 95, 84, 90, 74],
})

corr = students.corr(numeric_only=True).round(2)
print(corr)
print()
print("study_hours vs score:", corr.loc["study_hours", "score"], "-> strong positive:")
print("the students who study more, score more (no surprise, but now we can SHOW it)")

## 9. Declarative Filtering: `.query()`

Instead of building masks with brackets, `.query("...")` takes a **string expression** using column names directly — very readable for compound conditions. Text values inside the expression need their own quotes, so the outer quotes must differ.

⚠️ **The quoting gotcha:** `"city == 'Dhaka'"` works (double outside, single inside); `'city == 'Dhaka''` is broken — the string ends at the first inner quote. Reference outside Python variables by prefixing `@`. Column names containing spaces cannot be written bare — wrap them in backticks: ``df.query("`unit price` > 50")``.

**Syntax:**

```python
df.query("age > 20 and city == 'Dhaka'")   # and / or / not, in / not in all work
min_age = 20
df.query("age >= @min_age")                # @ pulls in an outside variable
```

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":        ["Sarah", "Rafi", "Nabila", "Karim", "Mim", "Tanvir", "Jony"],
    "age":         [22, 25, 21, 23, 20, 24, 22],
    "city":        ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna", "Dhaka", "Sylhet"],
    "study_hours": [2, 4, 1, 5, 3, 4, 2],
    "score":       [88, 92, 79, 95, 84, 90, 74],
})

print(students.query("age >= 21 and city == 'Dhaka'"))     # compound condition
print()
print(students.query("city in ['Dhaka', 'Sylhet']")["name"].tolist())
print()
min_score = 90                                             # an ordinary Python variable
print(students.query("score >= @min_score")[["name", "score"]])

## 10. Method Chaining: Reading Like a Recipe

Real analyses chain steps: filter → pick columns → sort → take top rows. Wrapped in parentheses a chain breaks across lines, one verb per line, and reads top-down like a recipe. This style scales far better than nested one-liners.

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "name":        ["Sarah", "Rafi", "Nabila", "Karim", "Mim", "Tanvir", "Jony"],
    "age":         [22, 25, 21, 23, 20, 24, 22],
    "city":        ["Dhaka", "Sylhet", "Dhaka", "Chattogram", "Khulna", "Dhaka", "Sylhet"],
    "study_hours": [2, 4, 1, 5, 3, 4, 2],
    "score":       [88, 92, 79, 95, 84, 90, 74],
})

report = (
    students
    .query("city == 'Dhaka'")            # 1. keep Dhaka students
    .loc[:, ["name", "study_hours", "score"]]   # 2. keep useful columns
    .sort_values("score", ascending=False)      # 3. best first
    .reset_index(drop=True)                     # 4. tidy row labels 0..n-1
)
print(report)

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| `df.mean()` on mixed data | pandas 3 raises `TypeError` instead of silently skipping text | Pass `numeric_only=True`, or select numeric columns first |
| Wrong quoting inside `.query()` | `'city == 'Dhaka''` ends the string at the first inner quote | Alternate quote styles: `"city == 'Dhaka'"` |
| Confusing `count()` with `size` | `size` counts every cell; `count()` skips NaN — numbers disagree | Ask which question you mean; compare `count().sum()` vs `size` |
| Full sort when you need top-N | Wastes time and invites `.head()` off-by-one slips | Use `nlargest(n, col)` / `nsmallest(n, col)` |
| Over-reading correlation | High r is not causation, and curves score near 0 | Plot the data; treat r as a hint, not proof |

## 💡 Best Practices & Pro Tips

- **Run `describe(include="all")` before any modeling.** Impossible minimums (`age = -3`) and odd `freq` values expose data-entry bugs in the first minute.
- **Prefer proportions:** `value_counts(normalize=True)` compares categories fairly across datasets of different sizes.
- Keep chains to **one operation per line**, wrapped in parentheses; name intermediate results when they get long.
- Seed anything random (`sample(random_state=42)`) so your analysis is reproducible tomorrow.
- 🤖 **AI-engineering relevance:** this toolkit *is* step one of every ML project — class balance checks are `value_counts(normalize=True)`, feature skew shows up in `describe()`, and `corr()` is the first filter against redundant features that add noise to models.

## 📌 Summary

| Method | What it does | Example |
|---|---|---|
| `describe(include="all")` | Statistical profile per column | `students.describe()` |
| `value_counts(normalize=True)` | Frequency table / proportions | `students["city"].value_counts()` |
| `unique()` / `nunique()` | Distinct values / their count | `students["city"].nunique()` |
| `mean/median/std/sum/quantile` | Column statistics | `students.mean(numeric_only=True)` |
| `count()` vs `size` | Non-nulls per column vs total cells | `messy.count()`, `messy.size` |
| `sort_values(by=[...], ascending=[...])` | Multi-key ordering | `sort_values(["city", "score"], ascending=[True, False])` |
| `nlargest` / `nsmallest` | Top-N / bottom-N rows | `students.nlargest(3, "score")` |
| `corr(numeric_only=True)` | Pairwise correlations (-1..+1) | `students.corr(numeric_only=True)` |
| `query("expr")` | String-based row filtering | `students.query("age > 20 and city == 'Dhaka'")` |

**Key takeaways**

- Profile first (`info` → `describe`), compute second — most surprises live in the profile.
- pandas 3 makes you say what you mean: reductions on mixed tables require `numeric_only=True`.
- Categorical questions use `value_counts`/`nunique`; ranking questions use `nlargest`/`nsmallest`.
- `.query()` reads like English — mind the quote styles, and use `@var` for outside values.

## 🔗 Next Lesson

Next up: **[05_Cleaning_Data](../05_Cleaning_Data/notes.ipynb)** — real datasets arrive broken; learn the repair toolkit for missing values, duplicates and messy text.